# Ejercicio de penetración indoor en un edificio real

Este cuaderno guía el análisis de cobertura indoor para voz, mensajería y acceso a plataformas digitales en un edificio de varias plantas. Se trabajan los siguientes pasos:

1. Definir puntos de usuario (entrada, aula 2ª planta, semisótano/archivo).
2. Calcular pérdida de propagación exterior (outdoor) desde una antena exterior a 20-25 m.
3. Calcular pérdidas por penetración: fachada, paredes interiores y forjados.
4. Calcular la pérdida total y el nivel recibido en cada punto.
5. Comprobar si el nivel recibido supera umbrales de servicio para voz y datos.
6. Proponer soluciones de mejora (DAS, repetidor, small cell) si no hay cobertura adecuada.
7. Generar tabla de pérdidas por tramo y un croquis del edificio con los puntos.
8. (Opcional) Comparar el efecto de cambiar la portadora a 700/800/1800/2100/2600 MHz.

> **Nota:** Las fórmulas y valores de pérdidas son ejemplos para un ejercicio de clase. En un análisis real se deben utilizar datos de antena, altura, topografía y mediciones reales.


In [ ]:
# Indoor penetration analysis for a multi-story building
# This code follows the exercise structure and computes outdoor + indoor losses,
# received power, and checks coverage for voice/data.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- 0) Parámetros de la estación base / antena exterior ----
P_tx_dBm = 43  # Potencia de transmisión típica (dBm)
G_tx_dBi = 15  # Ganancia de la antena exterior (dBi)
L_cable_dB = 2  # Pérdidas de cable/conector (dB)

# Umbrales de servicio (valores de referencia)
threshold_voice_dBm = -95  # Umbral aproximado para voz (e.g., GSM/UMTS)
threshold_data_dBm = -85   # Umbral aproximado para datos básicos (e.g., 3G/4G)

# Modelo de pérdida en espacio libre (dB)
def free_space_path_loss(freq_hz: float, dist_m: float) -> float:
    """Retorna la pérdida en espacio libre en dB."""
    c = 3e8
    lam = c / freq_hz
    if dist_m <= 0:
        return 0.0
    return 20 * np.log10(4 * np.pi * dist_m / lam)

# Puntos de usuario definidos en el edificio
user_points = {
    "Entrada": {
        "description": "Acceso principal a planta baja",
        "distancia_m": 100.0,  # distancia aproximada hasta la antena exterior
        "fachada_dB": 15.0,
        "paredes_dB": 4.0,
        "forjado_dB": 0.0,  # en planta baja no hay forjado por encima
    },
    "Aula 2ª planta": {
        "description": "Aula en segunda planta",
        "distancia_m": 120.0,
        "fachada_dB": 15.0,
        "paredes_dB": 8.0,  # paso por paredes interiores
        "forjado_dB": 15.0,  # penetración de forjado
    },
    "Semisótano/Archivo": {
        "description": "Semisótano técnico/archivo",
        "distancia_m": 80.0,
        "fachada_dB": 18.0,
        "paredes_dB": 10.0,
        "forjado_dB": 18.0,
    },
}

# Selección de portadora (por defecto LTE 1800)
carrier_freqs = {
    "700 MHz": 700e6,
    "800 MHz": 800e6,
    "1800 MHz": 1800e6,
    "2100 MHz": 2100e6,
    "2600 MHz": 2600e6,
}

# Seleccionar portadora base para análisis
carrier_label = "1800 MHz"
carrier_freq = carrier_freqs[carrier_label]

# Calculamos pérdidas y potencias recibidas para cada punto
def analyze_point(point: dict, freq_hz: float):
    dist = point["distancia_m"]
    L_outdoor = free_space_path_loss(freq_hz, dist)
    L_penetration = point["fachada_dB"] + point["paredes_dB"] + point["forjado_dB"]
    L_total = L_outdoor + L_penetration + L_cable_dB - G_tx_dBi  # Se resta la ganancia de la antena

    P_rx_dBm = P_tx_dBm - L_total

    estado_voice = "OK" if P_rx_dBm >= threshold_voice_dBm else "Insuficiente"
    estado_data = "OK" if P_rx_dBm >= threshold_data_dBm else "Insuficiente"

    return {
        "distancia_m": dist,
        "L_outdoor_dB": L_outdoor,
        "L_penetracion_dB": L_penetration,
        "L_total_dB": L_total,
        "P_rx_dBm": P_rx_dBm,
        "voice_ok": estado_voice,
        "data_ok": estado_data,
    }

results = {name: analyze_point(point, carrier_freq) for name, point in user_points.items()}

# Tabla de pérdidas
df = pd.DataFrame(results).T

# Añadimos recomendaciones simples
recommendations = {}
for name, row in df.iterrows():
    if row["P_rx_dBm"] < threshold_data_dBm:
        recommendations[name] = "Instalar DAS / small cell o repetidor."
    elif row["P_rx_dBm"] < threshold_voice_dBm:
        recommendations[name] = "Evaluar repetidor o mejora de antena para voz."
    else:
        recommendations[name] = "Cobertura macro suficiente."

df["recomendacion"] = pd.Series(recommendations)

# Mostrar resultados principales
print(f"Análisis de cobertura para portadora {carrier_label}")
print(df[["distancia_m", "L_outdoor_dB", "L_penetracion_dB", "L_total_dB", "P_rx_dBm", "recomendacion"]])

# Dibujar croquis sencillo del edificio
fig, ax = plt.subplots(figsize=(6, 6))
ax.set_title("Croquis simplificado del edificio y puntos de medición")
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)

# Dibujar contornos de plantas
ax.add_patch(plt.Rectangle((1, 1), 8, 8, fill=False, edgecolor="black"))
ax.text(1.5, 8.5, "2ª Planta", fontsize=10)
ax.text(1.5, 5.0, "Planta Baja", fontsize=10)
ax.text(1.5, 1.5, "Semisótano", fontsize=10)

# Añadir puntos de usuario (coordenadas aproximadas)
points_coords = {
    "Entrada": (3, 5.5),
    "Aula 2ª planta": (6.5, 8.0),
    "Semisótano/Archivo": (6, 2.0),
}

for name, (x, y) in points_coords.items():
    ax.plot(x, y, "ro")
    ax.text(x + 0.2, y + 0.2, name, fontsize=9)

ax.axis("off")
plt.show()

# Comparación opcional de frecuencias (700/800/1800/2100/2600)
comparison = []
for label, freq in carrier_freqs.items():
    for name, point in user_points.items():
        res = analyze_point(point, freq)
        comparison.append({
            "Portadora": label,
            "Punto": name,
            "P_rx_dBm": res["P_rx_dBm"],
        })

comp_df = pd.DataFrame(comparison)
pivot = comp_df.pivot(index="Punto", columns="Portadora", values="P_rx_dBm")

print("\nComparativa de nivel recibido según portadora (dBm):")
print(pivot)
